# IND2622 — Experimento I (Q1): versão final

**Q1.** O perfil setorial de uso de energia (MECS), ligado a cada família de intervenção pela relação semântica, melhora a priorização das famílias ARC (motores, HVAC, resfriamento) recomendadas nas avaliações ITAC?

Mudanças em relação à Etapa 2:

1. **XGBoost** como terceiro algoritmo (sugestão da professora), ao lado de Random Forest e regressão logística.
2. **Validação por subsetor deixando um setor de fora por vez** (17 partições, uma por NAICS3), no lugar de 5 grupos. Cada setor é testado sozinho, sem nunca ter aparecido no treino, e a comparação entre cenários passa a ter 17 pares.
3. **Diagnóstico setor a setor**: mostra que o MECS não reordena avaliações dentro de um setor; o ganho vem da ordenação entre setores, medida pelo bootstrap de setores.
4. **Efeito prático no ranking**: em quantas avaliações a família apontada em 1º lugar muda de A para C, e se a mudança acerta mais ou menos.

5. **Seed fixa (0)** em todas as execuções, por orientação da professora.

Mantidos da Etapa 2: ajuste de hiperparâmetros num corte dentro do treino (FY ≤ 2016 → 2017–2019), probabilidades sem reponderação de classes, controles aleatório, estatístico e placebo, métricas AP/macro-AP, LRAP e Recall@1.

Entradas (mesma pasta): `ITAC_MECS_2022_completecase_17_setores.csv` e `mecs_naics3_profile_audited.csv`.

Interface: `mlflow ui --backend-store-uri sqlite:///mlflow.db`

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, label_ranking_average_precision_score,
                             precision_recall_curve)
from sklearn.model_selection import LeaveOneGroupOut
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

DATA = Path("ITAC_MECS_2022_completecase_17_setores.csv")
MECS = Path("mecs_naics3_profile_audited.csv")
SAIDAS = Path("saidas"); SAIDAS.mkdir(exist_ok=True)

2026/11/22 16:26:01 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /usr/local/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Dados de entrada e checagens

A base já chega filtrada (manufatura, FY 2005–2025, NAICS de seis dígitos), unida ao MECS por NAICS3 e restrita aos 17 subsetores com as cinco participações elétricas publicadas. As checagens abaixo documentam o estado da base que entra no experimento.

In [2]:
df = pd.read_csv(DATA)
mecs = pd.read_csv(MECS)

FAMILIAS = ["motores", "hvac", "resfriamento"]
NUM_LOG = ["sales_usd_year", "employees", "plant_area_sqft", "production_hours_year",
           "electricity_kwh_year", "electricity_cost_usd_year"]
NUM_RAW = ["demand_cost_usd_year"]
CAT = ["naics3", "state"]
SHARES = ["share_machine_drive", "share_process_heating",
          "share_process_cooling_and_refrigeration", "share_facility_hvacf",
          "share_facility_lighting"]
SEMANTICA = {
    "motores": "share_machine_drive",
    "hvac": "share_facility_hvacf",
    "resfriamento": "share_process_cooling_and_refrigeration",
}

print("avaliações:", len(df), "| subsetores:", df.naics3.nunique(),
      "| FY:", df.assessment_fy.min(), "-", df.assessment_fy.max())
print("IDs duplicados:", df.assessment_id.duplicated().sum())
print("participações MECS ausentes:", int(df[SHARES].isna().sum().sum()))
print("valores distintos de cada participação por subsetor (máx):",
      int(df.groupby("naics3")[SHARES].nunique().max().max()))

avaliações: 8832 | subsetores: 17 | FY: 2005 - 2025
IDs duplicados: 0
participações MECS ausentes: 0
valores distintos de cada participação por subsetor (máx): 1


In [3]:
checagem = pd.DataFrame({
    "ausentes": df[NUM_LOG + NUM_RAW].isna().sum(),
    "% ausente": (100 * df[NUM_LOG + NUM_RAW].isna().mean()).round(2),
    "mín": df[NUM_LOG + NUM_RAW].min().round(2),
    "mediana": df[NUM_LOG + NUM_RAW].median().round(2),
    "máx": df[NUM_LOG + NUM_RAW].max().round(2),
})
checagem

,ausentes,% ausente,mín,mediana,máx
sales_usd_year,20,0.23,0.69,17.37,23.90
employees,2,0.02,0.69,4.80,9.35
plant_area_sqft,2,0.02,4.91,11.74,19.80
production_hours_year,1,0.01,0.69,8.70,11.26
electricity_kwh_year,12,0.14,3.61,15.26,20.92
electricity_cost_usd_year,9,0.10,4.45,12.49,18.29
demand_cost_usd_year,1197,13.55,0.00,93733.00,11851287.00


Os seis atributos de `NUM_LOG` já estão em log1p (medianas entre 8 e 16); `demand_cost_usd_year` ainda está em US$ e recebe log1p dentro do pipeline. Só `demand_cost_usd_year` tem ausência relevante (≈ 13,5%), tratada com mediana + indicador de ausência.

In [4]:
rotulos = pd.DataFrame({
    "positivos": df[FAMILIAS].sum(),
    "prevalência": df[FAMILIAS].mean().round(3),
})
print(rotulos, "\n")
print("famílias por avaliação:", df[FAMILIAS].sum(axis=1).value_counts().sort_index().to_dict())
print("\ncorrelação entre rótulos:\n", df[FAMILIAS].corr().round(2))

              positivos  prevalência
motores            4567        0.517
hvac               3370        0.382
resfriamento       1029        0.117 

famílias por avaliação: {0: 2299, 1: 4343, 2: 1947, 3: 243}

correlação entre rótulos:
               motores  hvac  resfriamento
motores          1.00 -0.05          0.05
hvac            -0.05  1.00          0.04
resfriamento     0.05  0.04          1.00


In [5]:
perfil = (df.groupby("naics3")
            .agg(n=("assessment_id", "size"), **{f: (f, "mean") for f in FAMILIAS},
                 **{s.replace("share_", ""): (s, "first") for s in SEMANTICA.values()})
            .round(3))
perfil

,n,motores,hvac,resfriamento,machine_drive,facility_hvacf,process_cooling_and_refrigeration
naics3,,,,,,,
311,1108,0.559,0.296,0.183,0.446,0.081,0.257
312,281,0.520,0.327,0.256,0.451,0.098,0.235
313,114,0.579,0.316,0.114,0.600,0.167,0.067
321,402,0.639,0.192,0.005,0.671,0.053,0.013
322,320,0.578,0.325,0.069,0.745,0.061,0.021
323,225,0.484,0.484,0.116,0.515,0.152,0.061
324,105,0.638,0.219,0.019,0.827,0.030,0.046
325,674,0.571,0.335,0.157,0.510,0.066,0.091
326,944,0.600,0.394,0.209,0.477,0.132,0.099


In [6]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, fam in zip(ax, FAMILIAS):
    a.scatter(perfil[SEMANTICA[fam].replace("share_", "")], perfil[fam], s=perfil["n"] / 8, alpha=.7)
    for nx, row in perfil.iterrows():
        a.annotate(str(nx), (row[SEMANTICA[fam].replace("share_", "")], row[fam]), fontsize=7)
    a.set_xlabel(SEMANTICA[fam].replace("share_", "") + " (MECS)")
    a.set_ylabel(f"taxa de recomendação {fam}")
fig.tight_layout()
fig.savefig(SAIDAS / "perfil_semantico_por_subsetor.png", dpi=150)
plt.close(fig)

print("correlação entre participação semântica e taxa de recomendação (17 subsetores):")
for fam in FAMILIAS:
    print(f"  {fam:13s}", round(perfil[SEMANTICA[fam].replace('share_', '')].corr(perfil[fam]), 2))

correlação entre participação semântica e taxa de recomendação (17 subsetores):
  motores       0.77
  hvac          0.85
  resfriamento  0.79


Ponto importante para a interpretação: **as participações MECS são constantes dentro de cada subsetor** (17 valores distintos). Logo:

- na **validação temporal**, o one-hot de NAICS3 já carrega toda a informação de subsetor e as participações não podem acrescentar nada além de uma recodificação;
- na **validação por subsetor**, o NAICS3 do teste nunca foi visto (o one-hot fica zerado) e a participação MECS é a única informação setorial disponível. É nessa validação que a Q1 é testada de fato.

Qualquer efeito do MECS se apoia em 17 subsetores, não em 8.832 avaliações independentes.

## 2. Plano do experimento

| Fator | Níveis |
|---|---|
| Cenário de atributos | B0 frequência, A tabular, B externo, C semântico, controle aleatório, controle estatístico, controle placebo |
| Algoritmo | Random Forest, XGBoost, regressão logística (L2) |
| Validação | por subsetor (deixa um NAICS3 de fora por vez, 17 partições) e temporal (treino FY ≤ 2019, teste FY ≥ 2020) |
| Seed | 0, fixa (orientação da professora) |

- **B0**: taxa da família no subsetor, calculada no treino (na validação por subsetor vira a taxa global).
- **A**: atributos da avaliação ITAC.
- **B**: A + as cinco participações MECS.
- **C**: A + a participação ligada à família pela relação semântica.
- **Controle aleatório**: A + uma participação sorteada entre as quatro **não** semânticas.
- **Controle estatístico**: A + a participação com maior |correlação| com o rótulo no treino (pode coincidir com C; a coincidência é registrada).
- **Controle placebo**: A + a participação semântica de C, mas tomada de outro subsetor (permutação fixa já presente na base). Separa o efeito do perfil setorial verdadeiro do efeito de “ter mais uma variável de nível setorial”.

Métricas: AP por família e macro-AP (principal); LRAP e Recall@1 para o ranking entre famílias, calculados nas avaliações com pelo menos uma recomendação; Recall@2 mantido só para comparação com a versão anterior.

In [7]:
CENARIOS = ["B0", "A", "B", "C", "ctrl_aleatorio", "ctrl_estatistico", "ctrl_placebo"]
ALGORITMOS = ["rf", "xgb", "logit"]
VALIDACOES = ["subsetor", "temporal"]
SEEDS = [0]

GRID = {
    "rf": [{"min_samples_leaf": l, "max_features": m} for l in (5, 20, 50) for m in ("sqrt", 0.5)],
    "xgb": [{"max_depth": d, "learning_rate": lr} for d in (3, 5) for lr in (0.03, 0.1)],
    "logit": [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)],
}


def montar_pipeline(algoritmo, extras, params, seed):
    if algoritmo in ("rf", "xgb"):
        num = SimpleImputer(strategy="median", add_indicator=True)
        num_raw = Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                            ("imp", SimpleImputer(strategy="median", add_indicator=True))])
        if algoritmo == "rf":
            modelo = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed, **params)
        else:
            modelo = XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                   eval_metric="logloss", n_jobs=2, random_state=seed, **params)
    else:
        num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                        ("esc", StandardScaler())])
        num_raw = Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                            ("esc", StandardScaler())])
        modelo = LogisticRegression(max_iter=2000, **params)
    pre = ColumnTransformer([
        ("num", num, NUM_LOG + extras),
        ("num_raw", num_raw, NUM_RAW),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), CAT),
    ])
    return Pipeline([("pre", pre), ("modelo", modelo)])


def atributos_extras(cenario, familia, treino, seed):
    if cenario in ("B0", "A"):
        return []
    if cenario == "B":
        return SHARES
    if cenario == "C":
        return [SEMANTICA[familia]]
    if cenario == "ctrl_aleatorio":
        opcoes = [s for s in SHARES if s != SEMANTICA[familia]]
        rng = np.random.default_rng(seed + 100 * FAMILIAS.index(familia))
        return [str(rng.choice(opcoes))]
    if cenario == "ctrl_estatistico":
        corr = treino[SHARES].corrwith(treino[familia]).abs()
        return [corr.idxmax()]
    if cenario == "ctrl_placebo":
        return ["placebo_" + SEMANTICA[familia]]
    raise ValueError(cenario)


def prever_b0(treino, teste, familia):
    taxa = treino.groupby("naics3")[familia].mean()
    return teste["naics3"].map(taxa).fillna(treino[familia].mean()).to_numpy()


def particoes(validacao):
    if validacao == "subsetor":
        yield from LeaveOneGroupOut().split(df, groups=df["naics3"])
    elif validacao == "temporal":
        yield np.where(df.assessment_fy <= 2019)[0], np.where(df.assessment_fy >= 2020)[0]
    elif validacao == "ajuste":
        yield np.where(df.assessment_fy <= 2016)[0], np.where(df.assessment_fy.between(2017, 2019))[0]


def metricas_ranking(Y, S):
    tem = Y.sum(axis=1) > 0
    Y, S = Y[tem], S[tem]
    top = np.argsort(-S, axis=1)
    rec1 = Y[np.arange(len(Y)), top[:, 0]].mean()
    acertos = sum(np.isin(np.flatnonzero(Y[i]), top[i, :2]).sum() for i in range(len(Y)))
    return {"LRAP": label_ranking_average_precision_score(Y, S),
            "recall_at_1": rec1, "recall_at_2": acertos / Y.sum()}


def macro_ap(Y, S):
    return np.mean([average_precision_score(Y[:, j], S[:, j]) for j in range(Y.shape[1])])

for v in ["subsetor", "temporal", "ajuste"]:
    tam = [len(te) for tr, te in particoes(v)]
    print(f"{v:9s}", len(tam), "partições | avaliações de teste por partição:", min(tam), "a", max(tam))

subsetor  17 partições | avaliações de teste por partição: 105 a 1277
temporal  1 partições | avaliações de teste por partição: 2661 a 2661
ajuste    1 partições | avaliações de teste por partição: 1211 a 1211


## 3. Ajuste de hiperparâmetros (sem tocar no teste)

Para cada algoritmo e cenário, o grid é avaliado no corte de ajuste (treino FY ≤ 2016, validação FY 2017–2019), que fica inteiramente dentro do treino temporal. O melhor conjunto é fixado e usado nas duas validações. Cada combinação é um run filho no MLflow.

In [8]:
mlflow.set_experiment("IND2622_Q1_etapa3_ajuste")
tr, va = next(particoes("ajuste"))
treino, valid = df.iloc[tr], df.iloc[va]
MELHORES, ajuste_log = {}, []

for alg in ALGORITMOS:
    for cenario in CENARIOS[1:]:
        with mlflow.start_run(run_name=f"ajuste|{alg}|{cenario}"):
            mlflow.log_params({"algoritmo": alg, "cenario": cenario,
                               "treino": "FY<=2016", "validacao": "FY2017-2019"})
            for params in GRID[alg]:
                with mlflow.start_run(run_name=str(params), nested=True):
                    S = np.zeros((len(valid), 3))
                    for j, fam in enumerate(FAMILIAS):
                        extras = atributos_extras(cenario, fam, treino, 0)
                        cols = NUM_LOG + NUM_RAW + CAT + extras
                        pipe = montar_pipeline(alg, extras, params, 0).fit(treino[cols], treino[fam])
                        S[:, j] = pipe.predict_proba(valid[cols])[:, 1]
                    m = macro_ap(valid[FAMILIAS].to_numpy(), S)
                    mlflow.log_params(params)
                    mlflow.log_metric("macro_AP_validacao", m)
                    ajuste_log.append({"algoritmo": alg, "cenario": cenario, **params, "macro_AP_val": m})
            melhor = max((r for r in ajuste_log if r["algoritmo"] == alg and r["cenario"] == cenario),
                         key=lambda r: r["macro_AP_val"])
            MELHORES[(alg, cenario)] = {k: melhor[k] for k in GRID[alg][0]}
            mlflow.log_params({"melhor_" + k: v for k, v in MELHORES[(alg, cenario)].items()})
            mlflow.log_metric("melhor_macro_AP_validacao", melhor["macro_AP_val"])

ajuste = pd.DataFrame(ajuste_log)
ajuste.to_csv(SAIDAS / "ajuste_hiperparametros.csv", index=False)
pd.DataFrame([{"algoritmo": a, "cenario": c, **p} for (a, c), p in MELHORES.items()])

2026/11/22 16:26:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/22 16:26:03 INFO mlflow.store.db.utils: Updating database tables


2026/11/22 16:26:04 INFO mlflow.tracking.fluent: Experiment with name 'IND2622_Q1_etapa3_ajuste' does not exist. Creating a new experiment.


,algoritmo,cenario,min_samples_leaf,max_features,max_depth,learning_rate,C
0,rf,A,5.0,sqrt,NaN,NaN,NaN
1,rf,B,5.0,sqrt,NaN,NaN,NaN
2,rf,C,5.0,sqrt,NaN,NaN,NaN
3,rf,ctrl_aleatorio,5.0,sqrt,NaN,NaN,NaN
4,rf,ctrl_estatistico,5.0,sqrt,NaN,NaN,NaN
5,rf,ctrl_placebo,5.0,sqrt,NaN,NaN,NaN
6,xgb,A,NaN,NaN,3.0,0.03,NaN
7,xgb,B,NaN,NaN,3.0,0.03,NaN
8,xgb,C,NaN,NaN,3.0,0.03,NaN
9,xgb,ctrl_aleatorio,NaN,NaN,3.0,0.03,NaN


## 4. Execução

Cada combinação algoritmo × validação × cenário × seed vira um run. As predições fora da amostra ficam guardadas para o pós-processamento (bootstrap e diferenças por partição).

In [9]:
mlflow.set_experiment("IND2622_Q1_etapa3")
dataset = mlflow.data.from_pandas(df, source=str(DATA), name="ITAC_MECS_2022_17_setores")

por_particao, predicoes, escolhas = [], {}, []

for alg in ALGORITMOS:
    for validacao in VALIDACOES:
        idx_teste = np.concatenate([te for _, te in particoes(validacao)])
        for cenario in CENARIOS:
            for seed in SEEDS:
                if cenario == "B0" and (alg != "rf" or seed > 0):
                    continue
                with mlflow.start_run(run_name=f"{alg}|{cenario}|{validacao}|s{seed}"):
                    mlflow.log_input(dataset, context="treino_teste")
                    params = {} if cenario == "B0" else MELHORES[(alg, cenario)]
                    mlflow.log_params({"algoritmo": "B0" if cenario == "B0" else alg,
                                       "cenario": cenario, "validacao": validacao, "seed": seed, **params})
                    mlflow.set_tags({"etapa": "3", "questao": "Q1"})

                    S_all = np.zeros((len(df), 3))
                    for k, (tr, te) in enumerate(particoes(validacao)):
                        treino, teste = df.iloc[tr], df.iloc[te]
                        for j, fam in enumerate(FAMILIAS):
                            if cenario == "B0":
                                S_all[te, j] = prever_b0(treino, teste, fam)
                                continue
                            extras = atributos_extras(cenario, fam, treino, seed)
                            if cenario in ("ctrl_estatistico", "ctrl_aleatorio"):
                                escolhas.append({"algoritmo": alg, "validacao": validacao, "cenario": cenario,
                                                 "seed": seed, "particao": k, "familia": fam,
                                                 "escolhida": extras[0],
                                                 "coincide_com_C": extras[0] == SEMANTICA[fam]})
                            cols = NUM_LOG + NUM_RAW + CAT + extras
                            pipe = montar_pipeline(alg, extras, params, seed).fit(treino[cols], treino[fam])
                            S_all[te, j] = pipe.predict_proba(teste[cols])[:, 1]
                        Yk = teste[FAMILIAS].to_numpy()
                        aps_k = {f: (average_precision_score(Yk[:, j], S_all[te, j]) if Yk[:, j].sum() > 0 else np.nan)
                                 for j, f in enumerate(FAMILIAS)}
                        mlflow.log_metric("macro_AP_particao", float(np.nanmean(list(aps_k.values()))), step=k)
                        por_particao.append({"algoritmo": alg, "validacao": validacao, "cenario": cenario,
                                             "seed": seed, "particao": k, "naics3": int(teste.naics3.iloc[0]) if validacao == "subsetor" else None,
                                             "macro_AP": np.nanmean(list(aps_k.values())),
                                             **{"AP_" + f: v for f, v in aps_k.items()}})

                    Y, S = df[FAMILIAS].to_numpy()[idx_teste], S_all[idx_teste]
                    predicoes[(alg, validacao, cenario, seed)] = S_all
                    metricas = {f"AP_{f}": average_precision_score(Y[:, j], S[:, j]) for j, f in enumerate(FAMILIAS)}
                    metricas["macro_AP"] = np.mean(list(metricas.values()))
                    metricas.update(metricas_ranking(Y, S))
                    mlflow.log_metrics(metricas)

                    fig, ax = plt.subplots(figsize=(5, 4))
                    for j, f in enumerate(FAMILIAS):
                        p, r, _ = precision_recall_curve(Y[:, j], S[:, j])
                        ax.plot(r, p, label=f"{f} (AP={metricas['AP_' + f]:.2f})")
                    ax.set_xlabel("Recall"); ax.set_ylabel("Precisão"); ax.legend(fontsize=8)
                    ax.set_title(f"{alg} | {cenario} | {validacao} | seed {seed}", fontsize=9)
                    mlflow.log_figure(fig, "curvas_pr.png"); plt.close(fig)

pp = pd.DataFrame(por_particao)
escolhas = pd.DataFrame(escolhas)
print("runs:", len(predicoes))

2026/11/22 16:29:53 INFO mlflow.tracking.fluent: Experiment with name 'IND2622_Q1_etapa3' does not exist. Creating a new experiment.


runs: 38


## 5. Pós-processamento

### 5.1 Resumo por cenário

In [10]:
runs = mlflow.search_runs(experiment_names=["IND2622_Q1_etapa3"])
cols = ["params.algoritmo", "params.validacao", "params.cenario", "metrics.macro_AP",
        "metrics.AP_motores", "metrics.AP_hvac", "metrics.AP_resfriamento",
        "metrics.LRAP", "metrics.recall_at_1", "metrics.recall_at_2"]
r = runs[cols].rename(columns=lambda c: c.split(".")[-1])
r["algoritmo"] = r["algoritmo"].replace({"B0": "-"})
resumo = r.groupby(["validacao", "algoritmo", "cenario"]).agg(["mean"]).round(3)
ordem = {c: i for i, c in enumerate(CENARIOS)}
resumo = resumo.sort_index(level=[0, 1, 2], key=lambda s: s.map(ordem) if s.name == "cenario" else s)
resumo.to_csv(SAIDAS / "resumo_cenarios.csv")
resumo.xs("mean", axis=1, level=1)

macro_AP  AP_motores  AP_hvac  AP_resfriamento   LRAP  recall_at_1  recall_at_2
validacao algoritmo cenario                                                                                          
subsetor  -         B0                   0.294       0.467    0.329            0.086  0.837        0.699        0.885
          logit     A                    0.474       0.695    0.528            0.198  0.859        0.745        0.885
                    B                    0.491       0.698    0.553            0.224  0.866        0.759        0.886
                    C                    0.487       0.697    0.551            0.214  0.863        0.754        0.884
                    ctrl_aleatorio       0.484       0.695    0.548            0.209  0.862        0.752        0.885
                    ctrl_estatistico     0.488       0.697    0.551            0.214  0.864        0.756        0.884
                    ctrl_placebo         0.465       0.696    0.520            0.179  0.859        0.747        0.884
          rf        A                    0.465       0.689    0.520            0.185  0.858        0.743        0.885
                    B                    0.485       0.689    0.543            0.221  0.863        0.753        0.887
                    C                    0.485       0.690    0.543            0.221  0.862        0.752        0.886
                    ctrl_aleatorio       0.474       0.689    0.543            0.189  0.861        0.749        0.885
                    ctrl_estatistico     0.485       0.691    0.543            0.221  0.866        0.760        0.886
                    ctrl_placebo         0.452       0.688    0.499            0.171  0.855        0.736        0.885
          xgb       A                    0.469       0.694    0.520            0.194  0.858        0.743        0.885
                    B                    0.497       0.698    0.555            0.237  0.865        0.757        0.888
                    C                    0.493       0.693    0.555            0.232  0.866        0.759        0.887
                    ctrl_aleatorio       0.484       0.696    0.551            0.204  0.861        0.749        0.887
                    ctrl_estatistico     0.495       0.698    0.555            0.232  0.866        0.760        0.887
                    ctrl_placebo         0.460       0.693    0.507            0.181  0.855        0.738        0.885
temporal  -         B0                   0.409       0.568    0.487            0.171  0.844        0.708        0.898
          logit     A                    0.490       0.669    0.586            0.214  0.853        0.729        0.897
                    B                    0.489       0.669    0.585            0.212  0.852        0.727        0.896
                    C                    0.489       0.670    0.585            0.213  0.852        0.728        0.896
                    ctrl_aleatorio       0.490       0.670    0.586            0.213  0.852        0.728        0.897
                    ctrl_estatistico     0.489       0.669    0.585            0.213  0.852        0.726        0.896
                    ctrl_placebo         0.489       0.669    0.586            0.213  0.853        0.729        0.897
          rf        A                    0.488       0.658    0.574            0.232  0.852        0.725        0.896
                    B                    0.486       0.649    0.574            0.234  0.853        0.730        0.894
                    C                    0.496       0.661    0.587            0.239  0.855        0.733        0.894
                    ctrl_aleatorio       0.492       0.657    0.585            0.234  0.853        0.729        0.896
                    ctrl_estatistico     0.494       0.656    0.587            0.239  0.858        0.740        0.894
                    ctrl_placebo         0.487       0.655    0.577            0.228  0.854        0.731        0.897
          xgb       A     

### 5.2 Diferença C − alternativa setor a setor (validação por subsetor)

Cada um dos 17 setores é uma partição. A tabela abaixo é um **diagnóstico**, não o teste da Q1: dentro de um setor de teste, a participação MECS tem o mesmo valor para todas as avaliações, então ela não tem como reordenar as avaliações **dentro** do setor. Na regressão logística, que não tem interações, a diferença C − A por setor fica praticamente em zero, como esperado. O efeito do MECS aparece ao ordenar avaliações **de setores diferentes**, que é o que a AP calculada sobre todas as avaliações juntas mede. Por isso a inferência da Q1 é o bootstrap por setor da seção 5.3.

In [11]:
sub = (pp[pp.validacao == "subsetor"]
       .groupby(["algoritmo", "naics3", "cenario"])["macro_AP"].mean().unstack("cenario"))
b0 = sub["B0"].dropna().droplevel(0)
dif_part, por_setor = [], []
for alg in ALGORITMOS:
    s_ = sub.loc[alg].copy()
    s_["B0"] = b0
    for outro in ["B0", "A", "B", "ctrl_aleatorio", "ctrl_estatistico", "ctrl_placebo"]:
        d = (s_["C"] - s_[outro]).dropna()
        dif_part.append({"algoritmo": alg, "comparacao": f"C - {outro}", "media_setores": round(d.mean(), 4),
                         "mediana_setores": round(d.median(), 4), "C_melhor_em": f"{(d > 0).sum()}/{len(d)}"})
        if outro == "A":
            por_setor.append(d.rename(alg))
dif_part = pd.DataFrame(dif_part)
dif_part.to_csv(SAIDAS / "diferencas_por_setor.csv", index=False)
por_setor = pd.concat(por_setor, axis=1).round(4)
por_setor.to_csv(SAIDAS / "C_menos_A_por_setor.csv")
dif_part

,algoritmo,comparacao,media_setores,mediana_setores,C_melhor_em
0,rf,C - B0,0.1313,0.1335,17/17
1,rf,C - A,-0.0013,0.0012,10/17
2,rf,C - B,0.0054,0.0015,10/17
3,rf,C - ctrl_aleatorio,0.0028,-0.0009,7/17
4,rf,C - ctrl_estatistico,0.0009,0.0006,11/17
5,rf,C - ctrl_placebo,0.0024,0.0040,13/17
6,xgb,C - B0,0.1439,0.1484,17/17
7,xgb,C - A,-0.0018,0.0013,10/17
8,xgb,C - B,0.0106,0.0018,10/17
9,xgb,C - ctrl_aleatorio,0.0076,-0.0002,8/17


In [12]:
por_setor.join(df.groupby("naics3").size().rename("n_avaliacoes"))

,rf,xgb,logit,n_avaliacoes
naics3,,,,
311.0,0.0035,0.0041,0.0006,1108
312.0,-0.0000,-0.0070,-0.0003,281
313.0,0.0056,0.0111,-0.0001,114
321.0,0.0028,0.0044,0.0004,402
322.0,-0.0151,-0.0012,0.0001,320
323.0,0.0012,0.0006,-0.0016,225
324.0,-0.0117,-0.0259,0.0000,105
325.0,-0.0124,0.0019,-0.0006,674
326.0,-0.0083,-0.0036,0.0014,944


### 5.3 Intervalos de confiança por bootstrap

As predições fora da amostra  são reamostradas mil vezes. Na validação por subsetor reamostram-se **subsetores inteiros** (17 grupos), porque o efeito do MECS está no nível do subsetor; na temporal, avaliações. O intervalo é o percentil 2,5–97,5 da diferença de macro-AP e de AP por família.

In [13]:
def media_seeds(alg, validacao, cenario):
    seeds = [0] if cenario == "B0" else SEEDS
    a = "rf" if cenario == "B0" else alg
    return np.mean([predicoes[(a, validacao, cenario, s)] for s in seeds], axis=0)


def bootstrap_dif(alg, validacao, outro, B=1000, seed=42):
    idx = np.concatenate([te for _, te in particoes(validacao)])
    Y = df[FAMILIAS].to_numpy()[idx]
    SC, SO = media_seeds(alg, validacao, "C")[idx], media_seeds(alg, validacao, outro)[idx]
    rng = np.random.default_rng(seed)
    if validacao == "subsetor":
        g = df["naics3"].to_numpy()[idx]
        membros = {u: np.flatnonzero(g == u) for u in np.unique(g)}
        chaves = list(membros)
        sorteio = lambda: np.concatenate([membros[u] for u in rng.choice(chaves, len(chaves))])
    else:
        sorteio = lambda: rng.integers(0, len(idx), len(idx))
    def difs(ii):
        d = [average_precision_score(Y[ii, j], SC[ii, j]) - average_precision_score(Y[ii, j], SO[ii, j])
             for j in range(3)]
        return [np.mean(d)] + d
    obs = difs(np.arange(len(idx)))
    boot = []
    for _ in range(B):
        ii = sorteio()
        if (Y[ii].sum(axis=0) == 0).any():
            continue
        boot.append(difs(ii))
    boot = np.array(boot)
    nomes = ["macro_AP"] + ["AP_" + f for f in FAMILIAS]
    return [{"algoritmo": alg, "validacao": validacao, "comparacao": f"C - {outro}", "metrica": n,
             "dif": round(obs[i], 4), "ic_inf": round(np.percentile(boot[:, i], 2.5), 4),
             "ic_sup": round(np.percentile(boot[:, i], 97.5), 4),
             "P(dif>0)": round((boot[:, i] > 0).mean(), 3)} for i, n in enumerate(nomes)]


ic = pd.DataFrame([linha for alg in ALGORITMOS for v in VALIDACOES
                   for outro in ["A", "B", "ctrl_aleatorio", "ctrl_estatistico", "ctrl_placebo"]
                   for linha in bootstrap_dif(alg, v, outro)])
ic.to_csv(SAIDAS / "bootstrap_ic.csv", index=False)
ic[ic.metrica == "macro_AP"]

,algoritmo,validacao,comparacao,metrica,dif,ic_inf,ic_sup,P(dif>0)
0,rf,subsetor,C - A,macro_AP,0.0202,0.0087,0.0306,1.000
4,rf,subsetor,C - B,macro_AP,0.0003,-0.0057,0.0039,0.542
8,rf,subsetor,C - ctrl_aleatorio,macro_AP,0.0114,0.0013,0.0183,0.989
12,rf,subsetor,C - ctrl_estatistico,macro_AP,-0.0002,-0.0017,0.0011,0.414
16,rf,subsetor,C - ctrl_placebo,macro_AP,0.0325,0.0160,0.0437,1.000
20,rf,temporal,C - A,macro_AP,0.0076,0.0016,0.0129,0.993
24,rf,temporal,C - B,macro_AP,0.0099,0.0050,0.0145,1.000
28,rf,temporal,C - ctrl_aleatorio,macro_AP,0.0040,-0.0009,0.0087,0.954
32,rf,temporal,C - ctrl_estatistico,macro_AP,0.0016,-0.0006,0.0037,0.921
36,rf,temporal,C - ctrl_placebo,macro_AP,0.0090,0.0037,0.0135,0.999


In [14]:
ic[(ic.validacao == "subsetor") & (ic.comparacao.isin(["C - A", "C - ctrl_placebo"]))]

,algoritmo,validacao,comparacao,metrica,dif,ic_inf,ic_sup,P(dif>0)
0,rf,subsetor,C - A,macro_AP,0.0202,0.0087,0.0306,1.000
1,rf,subsetor,C - A,AP_motores,0.0011,-0.0053,0.0071,0.635
2,rf,subsetor,C - A,AP_hvac,0.0228,0.0041,0.0383,0.994
3,rf,subsetor,C - A,AP_resfriamento,0.0367,0.0115,0.0580,0.998
16,rf,subsetor,C - ctrl_placebo,macro_AP,0.0325,0.0160,0.0437,1.000
17,rf,subsetor,C - ctrl_placebo,AP_motores,0.0028,-0.0031,0.0090,0.809
18,rf,subsetor,C - ctrl_placebo,AP_hvac,0.0445,0.0183,0.0715,1.000
19,rf,subsetor,C - ctrl_placebo,AP_resfriamento,0.0501,0.0182,0.0763,1.000
40,xgb,subsetor,C - A,macro_AP,0.0237,0.0129,0.0330,1.000
41,xgb,subsetor,C - A,AP_motores,-0.0011,-0.0073,0.0055,0.351


### 5.4 Quando o controle estatístico coincide com C

In [15]:
coinc = (escolhas[escolhas.cenario == "ctrl_estatistico"]
         .groupby(["validacao", "familia"])
         .agg(coincide=("coincide_com_C", "mean"),
              escolhida=("escolhida", lambda s: s.value_counts().index[0].replace("share_", "")))
         .round(2))
coinc.to_csv(SAIDAS / "coincidencia_ctrl_estatistico.csv")
coinc

coincide                          escolhida
validacao familia                                                  
subsetor  hvac               1.0                     facility_hvacf
          motores            0.0                     facility_hvacf
          resfriamento       1.0  process_cooling_and_refrigeration
temporal  hvac               1.0                     facility_hvacf
          motores            0.0                     facility_hvacf
          resfriamento       1.0  process_cooling_and_refrigeration

### 5.5 Efeito prático: a família apontada em 1º lugar muda?

Nas avaliações com pelo menos uma das três famílias recomendadas, compara-se a família colocada em 1º lugar por A e por C . Quando muda, verifica-se qual das duas escolhas foi de fato recomendada.

In [16]:
Yall = df[FAMILIAS].to_numpy()
linhas = []
for alg in ALGORITMOS:
    for v in VALIDACOES:
        idx = np.concatenate([te for _, te in particoes(v)])
        tem = Yall[idx].sum(axis=1) > 0
        Y = Yall[idx][tem]
        tA = media_seeds(alg, v, "A")[idx][tem].argmax(axis=1)
        tC = media_seeds(alg, v, "C")[idx][tem].argmax(axis=1)
        mud = tA != tC
        okA, okC = Y[np.arange(len(Y)), tA], Y[np.arange(len(Y)), tC]
        linhas.append({"algoritmo": alg, "validacao": v, "avaliacoes": len(Y),
                       "top1_muda_%": round(100 * mud.mean(), 1),
                       "so_C_acerta": int((mud & (okC == 1) & (okA == 0)).sum()),
                       "so_A_acerta": int((mud & (okA == 1) & (okC == 0)).sum()),
                       "recall1_A": round(okA.mean(), 3), "recall1_C": round(okC.mean(), 3)})
topo = pd.DataFrame(linhas)
topo.to_csv(SAIDAS / "mudanca_top1.csv", index=False)
topo

,algoritmo,validacao,avaliacoes,top1_muda_%,so_C_acerta,so_A_acerta,recall1_A,recall1_C
0,rf,subsetor,6533,10.2,259,201,0.743,0.752
1,rf,temporal,2041,6.4,57,40,0.725,0.733
2,xgb,subsetor,6533,11.5,327,221,0.743,0.759
3,xgb,temporal,2041,4.9,40,30,0.718,0.723
4,logit,subsetor,6533,11.7,306,246,0.745,0.754
5,logit,temporal,2041,1.6,7,10,0.729,0.728


In [17]:
sel = ic[(ic.validacao == "subsetor") & (ic.comparacao == "C - A")]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, alg in zip(axes, ALGORITMOS):
    d = sel[sel.algoritmo == alg].set_index("metrica")
    y = np.arange(len(d))
    ax.errorbar(d["dif"], y, xerr=[d["dif"] - d["ic_inf"], d["ic_sup"] - d["dif"]], fmt="o", capsize=3)
    ax.axvline(0, color="grey", lw=.8)
    ax.set_yticks(y, d.index)
    ax.set_title(f"C − A, subsetor ({alg})", fontsize=10)
    ax.set_xlabel("diferença de AP (IC 95%, bootstrap por subsetor)")
fig.tight_layout()
fig.savefig(SAIDAS / "ic_C_menos_A_subsetor.png", dpi=150)

mlflow.set_experiment("IND2622_Q1_etapa3")
with mlflow.start_run(run_name="pos_processamento_resumo"):
    mlflow.set_tags({"etapa": "3", "tipo": "resumo"})
    mlflow.log_figure(fig, "ic_C_menos_A_subsetor.png")
    mlflow.log_table(ic, "bootstrap_ic.json")
    mlflow.log_table(dif_part, "diferencas_por_setor.json")
    mlflow.log_table(topo, "mudanca_top1.json")
    mlflow.log_table(coinc.reset_index(), "coincidencia_ctrl_estatistico.json")
plt.close(fig)
pp.to_csv(SAIDAS / "metricas_por_particao.csv", index=False)